In [ ]:
import os
import re
import json
import pandas as pd

# Configuration: path to the directory containing your eval results
PARENT_DIR = "/"

# Compile regex to extract type, m, n, and l0
pattern = re.compile(
    r'^(?P<sae_type>topk|kronsae)(?:_(?P<m>\d+)_(?P<n>\d+))?_l0_(?P<l0>\d+)_custom_sae_eval_results'
)

records = []

for name in os.listdir(PARENT_DIR):
    path = os.path.join(PARENT_DIR, name)
    json_path = os.path.join(path, "eval_results.json") if os.path.isdir(path) else path

    if not os.path.exists(json_path) or not name.endswith(("_custom_sae_eval_results", ".json")):
        continue

    m = pattern.match(os.path.basename(name))
    if not m:
        continue

    gd = m.groupdict()
    sae_type = gd["sae_type"]
    l0       = int(gd["l0"])
    m_val    = gd.get("m")
    n_val    = gd.get("n")

    # Build model_name
    if m_val and n_val:
        # model_name = f"{sae_type}_{m_val}_{n_val}"
        model_name = f"KronSAE $m={m_val}$ $n={n_val}$"
    else:
        # model_name = sae_type
        model_name = "TopK"

    with open(json_path, 'r') as f:
        data = json.load(f)

    mean = data.get("eval_result_metrics", {}).get("mean", {})

    records.append({
        "name": name,
        "SAE": model_name,
        "sae_type": sae_type,
        "l0": l0,
        "mean_absorption_fraction_score": mean.get("mean_absorption_fraction_score"),
        "mean_full_absorption_score":     mean.get("mean_full_absorption_score"),
        "mean_num_split_features":        mean.get("mean_num_split_features"),
        "std_dev_absorption_fraction_score": mean.get("std_dev_absorption_fraction_score"),
        "std_dev_full_absorption_score":     mean.get("std_dev_full_absorption_score"),
        "std_dev_num_split_features":        mean.get("std_dev_num_split_features"),
    })

# Create DataFrame
df = pd.DataFrame(records)
print(df)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D 

sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(1, 3, figsize=(15+4, 5), dpi=250)

from matplotlib.pyplot import get_cmap
tbl_cb10 = get_cmap("tab10").colors  

sns.set_theme(style="whitegrid")
sns.set_palette("tab10")
g = sns.barplot(
    data=df,
    x="l0", y="mean_absorption_fraction_score", hue="SAE",
    errorbar="sd", 
    legend = False, 
    alpha=.99, ax=axs[0]
)


g.set_xlabel('$\\ell_0$', fontsize=18)
g.set_ylabel('Mean Absorption Fraction Score', fontsize=18)

g.tick_params(axis='both', which='major', labelsize=14)
g.tick_params(axis='both', which='minor', labelsize=10)


g1 = sns.barplot(
    data=df, 
    x="l0", y="mean_full_absorption_score", hue="SAE",
    errorbar="sd", 
    legend = False, 
    alpha=.99,  ax=axs[1]
)

g1.tick_params(axis='both', which='major', labelsize=14)
g1.tick_params(axis='both', which='minor', labelsize=10)
g1.set_xlabel('$\\ell_0$', fontsize=18)
g1.set_ylabel('Mean Full Absorption Score', fontsize=18)
g2 = sns.barplot(
    data=df, 
    x="l0", y="mean_num_split_features", hue="SAE",
    errorbar="sd", 
    alpha=.99,  ax=axs[2]
)

g2.set_xlabel('$\\ell_0$', fontsize=18)
g2.set_ylabel('Mean Number of Feature Splits', fontsize=18)
g2.tick_params(axis='both', which='major', labelsize=14)
g2.tick_params(axis='both', which='minor', labelsize=10)

handles, labels = g2.get_legend_handles_labels()


empty_handle = Line2D(
    [0], [0], color="white", marker='', linestyle="", markersize=4, label=''
)

custom_handles =  [empty_handle] + handles[:-1] + [empty_handle]  + [handles[-1]]
custom_labels = ['KronSAE: ', '$m=2$ $n=8$', '$m=4$ $n=4$', '$m=2$ $n=16$' , '$m=2$ $n=4$', '$m=4$ $n=8$', 'SAE:', 'TopK']

g.legend(
    handles=custom_handles,
    labels=custom_labels,
    loc='upper right',
    fontsize=14,
)
g2.get_legend().remove()

plt.tight_layout()
plt.savefig(f'absorption_score.pdf')
plt.show()